In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Load data
df = pd.read_csv("../data/churn.csv")

print("Data loaded")
print(f"Shape: {df.shape}")

Data loaded
Shape: (7043, 21)


In [3]:
# Fix TotalCharges (was stored as string)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop the 11 rows where conversion failed
df = df.dropna(subset=['TotalCharges'])

print(f"Shape after cleaning: {df.shape}")
print(f"TotalCharges type: {df['TotalCharges'].dtype}")

Shape after cleaning: (7032, 21)
TotalCharges type: float64


In [4]:
# customerID is unique - would cause overfitting
df = df.drop("customerID", axis=1)

print("Dropped customerID")
print(f"Features remaining: {df.shape[1]}")

Dropped customerID
Features remaining: 20


In [5]:
# Convert Yes/No to 1/0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Target encoded")
print(f"Churn = 1: {df['Churn'].sum()} customers")
print(f"Churn rate: {df['Churn'].mean()*100:.1f}%")

Target encoded
Churn = 1: 1869 customers
Churn rate: 26.6%


In [6]:
# Find categorical columns (text data)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# One-hot encode (drop_first prevents dummy variable trap)
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"\nEncoding complete")
print(f"New shape: {df.shape}")
print(f"Total features: {df.shape[1] - 1} (excluding target)")

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Encoding complete
New shape: (7032, 31)
Total features: 30 (excluding target)


/var/folders/p7/_97syqfx47z_bj_v4h67nymh0000gn/T/ipykernel_48227/1390891336.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [7]:
# Features (X) = everything except Churn
# Target (y) = Churn only
X = df.drop("Churn", axis=1)
y = df["Churn"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Features: {X.columns.tolist()[:5]}... (showing first 5)")

Features shape: (7032, 30)
Target shape: (7032,)
Features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male']... (showing first 5)


In [8]:
# Split 80% train, 20% test
# stratify=y ensures same churn rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate: {y_test.mean()*100:.1f}%")

Training set: 5625 samples
Test set: 1407 samples

Training churn rate: 26.6%
Test churn rate: 26.6%


In [9]:
# StandardScaler: mean=0, std=1 for each feature
scaler = StandardScaler()

# Fit on TRAINING data only (no cheating!)
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using training parameters
X_test_scaled = scaler.transform(X_test)

print("Scaling complete")
print(f"Scaled features mean: {X_train_scaled.mean():.2e}")
print(f"Scaled features std: {X_train_scaled.std():.2f}")

Scaling complete
Scaled features mean: -2.19e-17
Scaled features std: 1.00


In [10]:
# Save scaler for future predictions
joblib.dump(scaler, "../models/scaler.pkl")

# Save feature names for API
feature_names = X.columns.tolist()
joblib.dump(feature_names, "../models/feature_names.pkl")

# Save train/test sets for modeling
np.save("../models/X_train_scaled.npy", X_train_scaled)
np.save("../models/X_test_scaled.npy", X_test_scaled)
np.save("../models/y_train.npy", y_train)
np.save("../models/y_test.npy", y_test)

print("All preprocessing artifacts saved to /models/")

All preprocessing artifacts saved to /models/


In [13]:
# verification

print("PREPROCESSING SUMMARY\n")
print(f"Original data shape: 7043 x 21")
print(f"Cleaned data shape: {df.shape[0]} x {df.shape[1]}")
print(f"Features after encoding: {X.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Training churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate: {y_test.mean()*100:.1f}%")
print("\n Ready for modeling!")

PREPROCESSING SUMMARY

Original data shape: 7043 x 21
Cleaned data shape: 7032 x 31
Features after encoding: 30
Training samples: 5625
Test samples: 1407
Training churn rate: 26.6%
Test churn rate: 26.6%

 Ready for modeling!
